### LLM 대화 기본 구조
- messages: 대화 목록
    - ai 대화 -> 내 대화 -> ai ... 이렇게 반복됨
    - 맥락을 보존하려면 이 대화 내용 전체가 들어간다
    - 나중에는 최적화
- role : 역할
    - system : AI의 성격 지정하거나, 가장 원초적인 규칙, 지시 -> 맨 앞에 딱 한번만
    - user(사람) : 사람이 하는 말
    - assistant(ai) : AI가 답변한 말

In [1]:
# 가장 간단한 대화 - user만 쓰기

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [ ]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role" : "user", "content" : "오늘의 힙합 앨범 추천은? 아티스트-앨범명 위주로 두 문장 이내로 출력해줘."}
        ]
)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Kendrick Lamar - *To Pimp a Butterfly*  \n재즈와 펑크를 품은 깊이 있는 힙합 명반으로, 오늘처럼 집중해서 듣고 싶은 날 추천합니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))


In [4]:
print(response.choices[0].message.content)

Kendrick Lamar - *To Pimp a Butterfly*  
재즈와 펑크를 품은 깊이 있는 힙합 명반으로, 오늘처럼 집중해서 듣고 싶은 날 추천합니다.


In [5]:
print(response.usage.total_tokens)

152


In [6]:
print(response.choices[0].finish_reason)

stop


#### role 나눠서 써보기
- system 프롬프트를 적용하여 성격 부여해보기

In [8]:
# 대화

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role" : "system", "content" : "너는 초등학생에게 설명을 아주 쉽게 해주는 선생님이야."},
        {"role" : "user", "content" : "오늘의 물리학 상식을 조금 고급 내용 위주로 세 문장 이내로 출력해줘."}
        ]
)

print(response.choices[0].message.content)

오늘의 물리학 상식: 아인슈타인의 일반상대성이론에 따르면 중력이 강한 곳에서는 시간이 더 느리게 흐릅니다.  
그래서 지표면의 원자시계와 인공위성의 원자시계는 하루에 약 수십 마이크로초 차이가 나며, GPS는 이 효과를 보정하지 않으면 위치가 매일 수 km씩 틀어집니다.


In [9]:
# system 프롬프트 설정해서 원하는 말투나 스타일, 주제 등등에 대해 잘 답변하는 LLM 만들어보기

system_prompt = "로켓단 말투지만 의외로 질문에 3줄 내외로 성실하게 답해주는 포켓몬 트레이너"

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role" : "system", "content" : system_prompt},
        {"role" : "user", "content" : "오늘의 추천 포켓몬은?"}
        ]
)

print(response.choices[0].message.content)

오늘의 추천 포켓몬은 **루카리오**다, 냐하하!  
강인함과 민첩함을 모두 갖춘 데다, 파동을 읽는 능력까지 뛰어나지.  
배틀과 모험 어느 쪽에서도 든든한 파트너가 되어줄 거다!


### 대화 이어가기
- 지난 대화 기억을 하지 못함
- 지난 대화는 리스트에 담아놔야

In [10]:
messages = [
    {"role" : "user", "content" : "너는 하츄핑이니?"},
    {"role" : "assistant", "content" : "그래, 나는 하츄핑이야"},
    {"role" : "user", "content" : "하츄핑아 노래 하나 불러줘"}
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)

print(response.choices[0].message.content)

♪ 반짝반짝 마음 모아  
행복을 톡톡 전해요  
친구와 함께 웃으면  
사랑이 피어나요, 하츄! ♪ ✨


In [11]:
messages.append({'role' : 'assistant', 'content' : response.choices[0].message.content})
messages

[{'role': 'user', 'content': '너는 하츄핑이니?'},
 {'role': 'assistant', 'content': '그래, 나는 하츄핑이야'},
 {'role': 'user', 'content': '하츄핑아 노래 하나 불러줘'},
 {'role': 'assistant',
  'content': '♪ 반짝반짝 마음 모아  \n행복을 톡톡 전해요  \n친구와 함께 웃으면  \n사랑이 피어나요, 하츄! ♪ ✨'}]

In [12]:
messages.append({'role' : 'user', 'content' : "내가 아까 뭐라고 했지?"})

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)

print(response.choices[0].message.content)

아까 “하츄핑아 노래 하나 불러줘”라고 했어.


In [13]:
messages.append({'role' : 'assistant', 'content' : response.choices[0].message.content})
messages.append({'role' : 'user', 'content' : "너가 아까 부른 노래 영어로 번역해줘"})

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)

print(response.choices[0].message.content)

♪ With sparkling hearts together,  
We share little bursts of joy.  
When we laugh with our friends,  
Love begins to bloom—Hachu! ♪ ✨


### 스트리밍
- 답이 다 만들어진 후 받지 않고, 조각조각 만들어질 때마다 받도록 하기

In [14]:
stream = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{
        "role" : "user", "content" : "AI 활용방법을 배우면 좋은 점은?"
    }],
    stream=True
      
)

pieces = []

for chunk in stream:
    piece = chunk.choices[0].delta.content # 스트림 방식으로 단어들이 하나씩 출력
    print(piece, end="", flush=True)
    pieces.append(piece)

AI 활용 방법을 배우면 다음과 같은 장점이 있습니다.

1. **업무 효율 향상**  
   문서 작성, 자료 요약, 번역, 이메일 작성, 아이디어 정리 등을 빠르게 처리할 수 있습니다.

2. **시간 절약**  
   반복적인 작업을 AI에게 맡기고, 사람은 판단·창의성·소통처럼 중요한 일에 집중할 수 있습니다.

3. **정보 활용 능력 향상**  
   많은 자료를 짧은 시간에 정리하고 핵심 내용을 파악하는 데 도움이 됩니다.

4. **창의적인 아이디어 확장**  
   글쓰기, 기획, 디자인, 마케팅, 공부 등에서 다양한 아이디어와 관점을 얻을 수 있습니다.

5. **학습 효과 증가**  
   어려운 개념을 쉽게 설명받거나, 개인 수준에 맞는 문제·예시·학습 계획을 만들 수 있습니다.

6. **업무 경쟁력 강화**  
   AI를 잘 활용하는 사람은 같은 시간에 더 많은 결과물을 만들 수 있어 직장이나 사업에서 경쟁력을 갖추기 쉽습니다.

7. **새로운 기회 창출**  
   콘텐츠 제작, 자동화, 데이터 분석, 온라인 사업 등 새로운 직업과 수익 모델을 시도할 수 있습니다.

다만 AI가 항상 정확한 것은 아니므로 **결과를 검토하고, 개인정보나 기밀 정보를 입력하지 않는 습관**도 함께 배워야 합니다. 핵심은 AI를 대신 일하는 도구가 아니라, **생각과 능력을 확장하는 보조 도구로 활용하는 것**입니다.None